In [1]:
from stepmix.stepmix import StepMix
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pickle

In [2]:
cluster_df = pd.read_csv("../../data/master_data/2016_to_2023_clustering_input_data.csv")

len(cluster_df)

85659

In [3]:
cluster_cols = ["Age9",
                "Gend3",
                "Eth7",
                "Disab2_POP",
                "Educ6",
                "NSSEC5",
                "IMD10",
                "WorkStat8",
                "Child4",
                "HHLiv9",
                "Motiva_POP",
                "motivd_POP"]

X = cluster_df[cluster_cols]

X.head()

,Age9,Gend3,Eth7,Disab2_POP,Educ6,NSSEC5,IMD10,WorkStat8,Child4,HHLiv9,Motiva_POP,motivd_POP
0,0,0,3,1,0,0,1,1,1,2,0,3
1,2,0,3,1,0,1,3,1,0,1,1,3
2,2,1,0,1,0,0,6,0,0,0,1,3
3,2,0,1,1,0,0,3,0,1,4,1,3
4,2,1,1,0,0,0,3,1,1,4,0,2


In [4]:
results = []

for k in [27]:

    try:
    
        print(f"Fitting {k} classes...")

        model = StepMix(n_components=k,  measurement="categorical", random_state=42, n_init=30, max_iter=5000, abs_tol=1e-5)

        model.fit(X)

        with open(f"stepmix_{k}_class_model.pkl", "wb") as f:
             pickle.dump(model, f)

        posterior = model.predict_proba(X)
        predicted = model.predict(X)

        proportions = posterior.mean(axis=0)
        assignment = posterior.max(axis=1)

        loglik = model.score(X) * len(X)

        print(f"Log-likelihood : {loglik:,.2f}")
        print(f"AIC            : {model.aic(X):,.2f}")
        print(f"BIC            : {model.bic(X):,.2f}")
        print(f"Converged      : {model.converged_}")
        print(f"Iterations     : {model.n_iter_}")

        print("\nClass proportions")
        print(pd.Series(proportions).round(4))

        print("\nObserved class sizes")
        print(pd.Series(predicted).value_counts(normalize=True).sort_index().round(4))

        print("\nPosterior assignment certainty")
        print(pd.Series(assignment).describe())

        print(f"\nMean assignment probability : {assignment.mean():.4f}")
        print(f"Median assignment           : {np.median(assignment):.4f}")
        print(f"Minimum assignment          : {assignment.min():.4f}")

        temp = cluster_df.copy()
        temp["Class"] = predicted

        print("\nYear by latent class")
        print(pd.crosstab(temp["year"], temp["Class"], normalize="index").round(3))

        print("\nModel attributes")
        print(sorted(model.__dict__.keys()))

        params = model.get_parameters()

        print("\nParameter keys:")
        print(params.keys())

        measurement = params["measurement"]

        print("\nMeasurement parameters")
        print(type(measurement))

        if isinstance(measurement, dict):
            print("Measurement keys:")
            print(measurement.keys())

            for key, value in measurement.items():
                print(f"\n{key}")
                print(type(value))
                if hasattr(value, "shape"):
                    print("Shape:", value.shape)
        else:
            if hasattr(measurement, "shape"):
                print("Shape:", measurement.shape)
            else:
                print(measurement)

        results.append({"Classes": k,
                        "LogLik": loglik,
                        "AIC": model.aic(X),
                        "BIC": model.bic(X),
                        "Converged": model.converged_,
                        "Iterations": model.n_iter_,
                        "MeanAssignment": assignment.mean(),
                        "MedianAssignment": np.median(assignment),
                        "MinClass": proportions.min(),
                        "MaxClass": proportions.max(),
                        "EffectiveClasses": (proportions > 0.01).sum()})
        
    except Exception as e:
        results.append({"Classes": k, "Error": str(e)})

    pd.DataFrame(results).to_csv("lca_candidate_models_results.csv", index=False)

Fitting 27 classes...
Fitting StepMix...


Initializations (n_init) : 100%|██████████| 30/30 [25:51<00:00, 51.71s/it, max_LL=-1.15e+6, max_avg_LL=-13.5]


Log-likelihood : -1,152,684.59
AIC            : 2,308,661.17
BIC            : 2,324,064.65
Converged      : True
Iterations     : 157

Class proportions
0     0.0254
1     0.0340
2     0.0875
3     0.0582
4     0.0327
5     0.0489
6     0.0232
7     0.0583
8     0.0192
9     0.0311
10    0.0471
11    0.0582
12    0.0398
13    0.0171
14    0.0197
15    0.0513
16    0.0321
17    0.0464
18    0.0073
19    0.0391
20    0.0612
21    0.0329
22    0.0353
23    0.0341
24    0.0174
25    0.0160
26    0.0264
dtype: float64

Observed class sizes
0     0.0237
1     0.0323
2     0.0908
3     0.0603
4     0.0344
5     0.0447
6     0.0194
7     0.0617
8     0.0187
9     0.0262
10    0.0444
11    0.0512
12    0.0419
13    0.0168
14    0.0192
15    0.0583
16    0.0308
17    0.0585
18    0.0066
19    0.0410
20    0.0595
21    0.0336
22    0.0352
23    0.0315
24    0.0155
25    0.0158
26    0.0280
Name: proportion, dtype: float64

Posterior assignment certainty
count    85659.000000
mean         0.741106

In [5]:
results_df = pd.DataFrame(results)

print(results_df)

   Classes        LogLik           AIC           BIC  Converged  Iterations  \
0       27 -1.152685e+06  2.308661e+06  2.324065e+06       True         157   

   MeanAssignment  MedianAssignment  MinClass  MaxClass  EffectiveClasses  
0        0.741106          0.765033  0.007258   0.08748                26  
